[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/laws_of_motion.ipynb)

# Newton's laws of motion full course: the practice set, in Python

Twenty-eight exercises in three tiers: **tier 1** warm-ups (one idea, one
step), **tier 2** core problems (the level of the course's "Problems, step by step" chapters) and **tier 3** stretch
problems (two ideas at once, a formula run backwards, a trap). Each tier is harder than the last.

Under every tier there's a short **how you'd do this in Python** cell: the same physics with Python's `math` module
and `numpy`, so you can run it on your own numbers. And the course's simulations, with sliders you can move yourself:
a puck sliding on ice, a ball caught by stiff and soft hands, a block sliding down a ramp, and the safe speeds on the
course's banked exit ramp. Each simulation steps Newton's second law forward in small time steps (the fourth-order
Runge–Kutta method) and is checked against the formula the course derives.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When sliders appear, move them and watch: the picture
redraws by itself. Change any number and run the cell again: nothing can break. It works in your browser (no
install) and in Google Colab.

**Typing answers.** Work the question out first (on paper, or in a cell of your own), then type your answer into the
box and press **check**. Answers to two decimals, in the unit the question asks for; you can type the unit too
(`70 N`, `2.41 m/s²`). Numbers without thousands separators: `2500`, not `2,500`. You can type the arithmetic:
`0.4*(10+15)/0.02`, and `sqrt(…)` works. For a choice, type the option's number. For several answers, separate them
with commas, in the order the question lists them. Press **show working** to see the steps. You can also put your
answer between the quotes in a cell's last line, `practice("P1.5", "")`, and run the cell: it checks straight away.

**The cheat sheet.** Every law, formula and example from the course on two pages:
[laws of motion cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/laws_of_motion_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look, the checker and the physics: run this cell first.
import ast, math, operator, re, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from ipywidgets import interact, FloatSlider, Text, Button, Output, HBox, VBox, Layout

SLATE = "#1f2b28"                                     # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # bodies and data in chalk, chalk smudge, ghost lines
SKY, VIOLET, CORAL, GOLD, MINT = "#6fb6bb", "#b49ad6", "#e3896b", "#e8c56b", "#a0c27c"
# velocity and speed SKY · acceleration MINT · force CORAL · energy VIOLET · bodies and distance CHALK · the number
# that matters GOLD

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def footnote(fig, text="simulated · our own set-up"):
    fig.text(0.99, 0.01, text, color=DIM, fontsize=10, ha="right", va="bottom", family="monospace")

# ---- reading a typed answer ----
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
       ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}

def calc(text):
    """Plain arithmetic only: numbers, + − × ÷, powers, brackets and sqrt(…)."""
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.operand))
        if isinstance(n, ast.Call) and getattr(n.func, "id", "") == "sqrt" and len(n.args) == 1: return math.sqrt(ev(n.args[0]))
        raise ValueError(text)
    return float(ev(ast.parse(text, mode="eval")))

UNITS = r"kg\s*m/s|N\s*s|m/s²|m/s\^2|m/s2|m/s|km/h|kg|newtons?|degrees?|°|\bN\b"

def read_part(s):
    s = s.strip().replace("−", "-").replace("–", "-").replace("×", "*").replace("÷", "/")
    t = re.sub(UNITS, "", s, flags=re.I).strip().replace("^", "**")
    try: return calc(t)
    except Exception: return " ".join(s.lower().split())

def right(got, want, tol):
    return not isinstance(got, str) and abs(got - want) <= max(tol, 1e-7)

def show(v):
    if isinstance(v, str): return v
    return f"{v:,.10g}".replace(",", " ").replace("-", "−")

# ---- the exercises: the practice set's own (the course's practice.py; the last cell checks they still match) ----
# @EX
EX = [
    {"id": 'P1.1', "tier": 1, "group": 'Force and the first law',
     "q": 'Two forces act on a box: 250 N to the right and 180 N to the left. What is the net force, in newtons (to the right is positive)?',
     "parts": ['net force (N)'],
     "answer": [70.0], "tol": [0.005], "function": '—',
     "working": '250 − 180 = 70 N, to the right'},
    {"id": 'P1.2', "tier": 1, "group": 'Force and the first law',
     "q": 'A van cruises at a steady 20 m/s on a straight, level road. Drag and road resistance push back with 450 N. How hard do the driving wheels push forward, in newtons?',
     "parts": ['forward push (N)'],
     "answer": [450.0], "tol": [0.005], "function": '—',
     "working": 'Steady velocity → net force zero (first law) → forward push = 450 N'},
    {"id": 'P1.3', "tier": 1, "group": 'Momentum, the second law and impulse',
     "q": 'What is the momentum of a 0.06 kg ball moving at 30 m/s, in kg m/s?',
     "parts": ['momentum (kg m/s)'],
     "answer": [1.7999999999999998], "tol": [0.005], "function": '—',
     "working": 'p = m v = 0.06 × 30 = 1.8 kg m/s'},
    {"id": 'P1.4', "tier": 1, "group": 'Momentum, the second law and impulse',
     "q": 'What is the weight of a 75 kg climber, in newtons (g = 9.8 m/s²)?',
     "parts": ['weight (N)'],
     "answer": [735.0], "tol": [0.005], "function": '—',
     "working": 'W = m g = 75 × 9.8 = 735.0 N'},
    {"id": 'P1.5', "tier": 1, "group": 'Momentum, the second law and impulse',
     "q": 'A net force of 3 N acts on a 0.5 kg toy car. What is its acceleration, in m/s²?',
     "parts": ['acceleration (m/s²)'],
     "answer": [6.0], "tol": [0.005], "function": '—',
     "working": 'a = F/m = 3/0.5 = 6 m/s²'},
    {"id": 'P1.6', "tier": 1, "group": 'Momentum, the second law and impulse',
     "q": 'A force of 50 N acts for 0.2 s. What impulse does it give, in N s?',
     "parts": ['impulse (N s)'],
     "answer": [10.0], "tol": [0.005], "function": '—',
     "working": 'J = F Δt = 50 × 0.2 = 10 N s'},
    {"id": 'P1.7', "tier": 1, "group": 'Free-body diagrams and equilibrium',
     "q": 'A 3 kg book rests on a level table. What is the normal force on it, in newtons?',
     "parts": ['normal force (N)'],
     "answer": [29.400000000000002], "tol": [0.005], "function": '—',
     "working": 'At rest: N = W = 3 × 9.8 = 29.4 N'},
    {"id": 'P1.8', "tier": 1, "group": 'Friction',
     "q": 'A box presses on the floor with a normal force of 200 N, and μs = 0.3. What is the largest static friction force, in newtons?',
     "parts": ['largest static friction (N)'],
     "answer": [60.0], "tol": [0.005], "function": '—',
     "working": 'f_max = μs N = 0.3 × 200 = 60 N'},
    {"id": 'P1.9', "tier": 1, "group": 'Circular motion',
     "q": 'A 2 kg ball moves at 3 m/s in a circle of radius 1.5 m. What net force points to the centre, in newtons?',
     "parts": ['net force to the centre (N)'],
     "answer": [12.0], "tol": [0.005], "function": '—',
     "working": 'F = m v²/r = 2 × 9/1.5 = 12 N'},
    {"id": 'P1.10', "tier": 1, "group": 'Circular motion',
     "q": 'What tilt (degrees) lets a car round a 50 m radius curve at 14 m/s with no friction at all?',
     "parts": ['tilt (°)'],
     "answer": [21.80140948635181], "tol": [0.005], "function": 'math.degrees, math.atan',
     "working": 'tan θ = v²/(r g) = 196/490 = 0.4 → θ = 21.80°. In Python: math.degrees(math.atan(14**2 / (50 * 9.8)))'},
    {"id": 'P2.1', "tier": 2, "group": 'Momentum, the second law and impulse',
     "q": 'A 1,000 kg car speeds up steadily from rest to 25 m/s in 10 s. What net force acts on it, in newtons?',
     "parts": ['net force (N)'],
     "answer": [2500.0], "tol": [0.005], "function": '—',
     "working": 'a = 25/10 = 2.5 m/s²; F = m a = 1,000 × 2.5 = 2,500 N'},
    {"id": 'P2.2', "tier": 2, "group": 'Momentum, the second law and impulse',
     "q": 'A 0.4 kg ball hits a wall at 15 m/s and bounces straight back at 10 m/s. The contact lasts 0.02 s. What is the average force of the wall on the ball, in newtons?',
     "parts": ['average force (N)'],
     "answer": [500.0], "tol": [0.005], "function": '—',
     "working": "Take 'away from the wall' as positive: Δp = 0.4 × (10 − (−15)) = 10 N s; F = 10/0.02 = 500 N. (Ignoring the direction gives 0.4 × 5 = 2 N s and 100 N: wrong)"},
    {"id": 'P2.3', "tier": 2, "group": 'The third law and momentum',
     "q": 'An 80 kg skater at rest throws a 2 kg ball forward at 10 m/s. How fast does the skater move backwards, in m/s?',
     "parts": ["skater's speed backwards (m/s)"],
     "answer": [0.25], "tol": [0.005], "function": '—',
     "working": '0 = 2 × 10 + 80 v → v = −0.25 m/s: 0.25 m/s backwards'},
    {"id": 'P2.4', "tier": 2, "group": 'The third law and momentum',
     "q": 'A 12,000 kg wagon at 3 m/s hits an 8,000 kg wagon at rest, and they couple. How fast do they move on, in m/s?',
     "parts": ['speed together (m/s)'],
     "answer": [1.8], "tol": [0.005], "function": '—',
     "working": '12,000 × 3 = 20,000 v → v = 1.8 m/s'},
    {"id": 'P2.5', "tier": 2, "group": 'Free-body diagrams and equilibrium',
     "q": 'An 8 kg lamp hangs from two cables, each at 40° above the horizontal. What is the tension in each, in newtons?',
     "parts": ['tension in each cable (N)'],
     "answer": [60.98437401292817], "tol": [0.005], "function": 'math.sin, math.radians',
     "working": '2 T sin 40° = 8 × 9.8 = 78.4 → T = 78.4/(2 × 0.6428) = 60.98 N. In Python: 8 * 9.8 / (2 * math.sin(math.radians(40)))'},
    {"id": 'P2.6', "tier": 2, "group": 'Free-body diagrams and equilibrium',
     "q": 'A 70 kg passenger stands on a scale in a lift accelerating upwards at 1.5 m/s². What does the scale read, in newtons?',
     "parts": ['scale reading (N)'],
     "answer": [791.0], "tol": [0.005], "function": '—',
     "working": 'N − m g = m a → N = 70 × (9.8 + 1.5) = 791 N'},
    {"id": 'P2.7', "tier": 2, "group": 'Free-body diagrams and equilibrium',
     "q": 'Two blocks touch on a smooth floor: 3 kg in front of 5 kg. A 24 N push acts on the 3 kg block, towards the 5 kg block. What force does the 3 kg block exert on the 5 kg block, in newtons?',
     "parts": ['force on the 5 kg block (N)'],
     "answer": [15.0], "tol": [0.005], "function": '—',
     "working": 'Both together: a = 24/8 = 3 m/s²; the 5 kg block alone: F = 5 × 3 = 15 N'},
    {"id": 'P2.8', "tier": 2, "group": 'Friction',
     "q": 'A 30 kg crate already sliding on a floor (μk = 0.35) is pushed with 150 N. What is its acceleration, in m/s²?',
     "parts": ['acceleration (m/s²)'],
     "answer": [1.5700000000000003], "tol": [0.005], "function": '—',
     "working": 'N = 294 N; f_k = 0.35 × 294 = 102.9 N; a = (150 − 102.9)/30 = 1.57 m/s²'},
    {"id": 'P2.9', "tier": 2, "group": 'Friction',
     "q": 'A box on a board starts to slide when the board is tilted to some angle. If μs = 0.35, what angle is that, in degrees?',
     "parts": ['sliding angle (°)'],
     "answer": [19.290046219188735], "tol": [0.005], "function": 'math.degrees, math.atan',
     "working": 'tan θ = μs → θ = atan 0.35 = 19.29°. In Python: math.degrees(math.atan(0.35))'},
    {"id": 'P2.10', "tier": 2, "group": 'Circular motion',
     "q": 'On a flat road, a car turns on a curve of radius 45 m, with μs = 0.6. What is the fastest it can go without skidding, in m/s?',
     "parts": ['top speed (m/s)'],
     "answer": [16.26653005407115], "tol": [0.005], "function": 'math.sqrt',
     "working": 'm v²/r ≤ μs m g → v = √(0.6 × 45 × 9.8) = 16.27 m/s (58.6 km/h). In Python: math.sqrt(0.6 * 45 * 9.8)'},
    {"id": 'P3.1', "tier": 3, "group": 'Force and the first law',
     "q": "A lift's cable pulls up with exactly the lift's weight, and the lift is moving down. Is it (1) speeding up, (2) slowing down, or (3) moving at a steady speed? Type 1, 2 or 3.",
     "parts": ['type 1, 2 or 3'],
     "answer": [3], "tol": [0], "function": '—',
     "working": "Net force zero → no change in velocity: it keeps moving down at a steady speed (3). The direction of motion doesn't matter; only the net force decides the change"},
    {"id": 'P3.2', "tier": 3, "group": 'Momentum, the second law and impulse',
     "q": 'A net force of 2,000 N gives a loaded trolley an acceleration of 0.8 m/s². What is its mass, in kg?',
     "parts": ['mass (kg)'],
     "answer": [2500.0], "tol": [0.005], "function": '—',
     "working": 'm = F/a = 2,000/0.8 = 2,500 kg'},
    {"id": 'P3.3', "tier": 3, "group": 'The third law and momentum',
     "q": 'A 90 kg astronaut floating at rest throws a 3 kg camera and drifts back at 0.15 m/s. How fast was the camera thrown, in m/s?',
     "parts": ["camera's speed (m/s)"],
     "answer": [4.5], "tol": [0.005], "function": '—',
     "working": '0 = 3 v − 90 × 0.15 → v = 13.5/3 = 4.5 m/s'},
    {"id": 'P3.4', "tier": 3, "group": 'Free-body diagrams and equilibrium',
     "q": 'A 20 kg sign hangs from two cables, one at 30° and one at 45° above the horizontal, on opposite sides. Find both tensions, in newtons (the 30° cable first).',
     "parts": ['tension, 30° cable (N)', 'tension, 45° cable (N)'],
     "answer": [143.48195828349998, 175.72879254493853], "tol": [0.005, 0.005], "function": 'math.sin, math.cos, math.radians',
     "working": 'ΣFx: T₁ cos 30° = T₂ cos 45°; ΣFy: T₁ sin 30° + T₂ sin 45° = 196 → T₁ = 143.48 N, T₂ = 175.73 N (the steeper cable carries more)'},
    {"id": 'P3.5', "tier": 3, "group": 'Free-body diagrams and equilibrium',
     "q": "A 50 kg student stands on a scale in a lift; it reads 450 N. What is the lift's acceleration, in m/s² (upwards positive)?",
     "parts": ['acceleration, upwards + (m/s²)'],
     "answer": [-0.8000000000000007], "tol": [0.005], "function": '—',
     "working": 'a = N/m − g = 9 − 9.8 = -0.8 m/s²: 0.8 m/s² downwards (it could be going down and speeding up, or going up and slowing down)'},
    {"id": 'P3.6', "tier": 3, "group": 'Friction',
     "q": 'A box slides down a 25° ramp, speeding up at 2.0 m/s². What is μk between the box and the ramp?',
     "parts": ['μk'],
     "answer": [0.2411284910197962], "tol": [0.005], "function": 'math.sin, math.cos, math.radians',
     "working": 'a = g (sin θ − μk cos θ) → μk = (9.8 sin 25° − 2)/(9.8 cos 25°) = 0.24'},
    {"id": 'P3.7', "tier": 3, "group": 'Circular motion',
     "q": 'A curve of radius 150 m is banked at 8°. At what speed (m/s) does a car need no friction at all?',
     "parts": ['speed (m/s)'],
     "answer": [14.373413895540455], "tol": [0.005], "function": 'math.sqrt, math.tan, math.radians',
     "working": 'v = √(r g tan θ) = √(150 × 9.8 × tan 8°) = 14.37 m/s (51.7 km/h)'},
    {"id": 'P3.8', "tier": 3, "group": 'Circular motion',
     "q": 'The same idea with friction: a curve of radius 100 m is banked at 6°, and μs = 0.3. What is the fastest safe speed, in m/s?',
     "parts": ['top speed (m/s)'],
     "answer": [20.246671121934746], "tol": [0.005], "function": 'math.sqrt, math.tan, math.radians',
     "working": 'v² = r g (tan θ + μs)/(1 − μs tan θ) = 980 × 0.4051/0.9685 → v = 20.25 m/s'},
]
# @END EX
Q = {e["id"]: e for e in EX}
TIER_NAMES = {1: "Warm-up", 2: "Core", 3: "Stretch"}
assert len(EX) == 28 and [sum(e["tier"] == t for e in EX) for t in TIER_NAMES] == [10, 10, 8]

def practice(k, answer=""):
    """Exercise k ("P1.4"): an answer box, a check button and a show-working button."""
    q = Q[k]
    n = len(q["answer"])
    box = Text(value=answer, placeholder="your answer" if n == 1 else f"{n} answers, separated by commas",
               layout=Layout(width="520px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            got = [read_part(p) for p in re.split(r"[,;]", box.value) if p.strip()]
            if len(got) != n:
                print(f"✗ this one asks for {n} answer(s), in this order: {' · '.join(q['parts'])}. You typed {len(got)}."); return
            ok = [right(g, w, t) for g, w, t in zip(got, q["answer"], q["tol"])]
            if all(ok): print("✓ right" if n == 1 else f"✓ all {n} right"); return
            for good, part in zip(ok, q["parts"]):
                print(f"  {'✓' if good else '✗'} {part}" + ("" if good else ": not quite"))
    def on_work(_=None):
        out.clear_output()
        with out:
            print(textwrap.fill(q["working"], 110))
            if q["function"] not in ("—", ""): print(textwrap.fill(f"Python functions: {q['function']}.", 110))
            print("Answer: " + " · ".join(f"{p}: {show(a)}" for p, a in zip(q["parts"], q["answer"])))
    check.on_click(on_check); work.on_click(on_work)
    print(textwrap.fill(f"{k}. {q['q']}", 110))
    if n > 1: print(textwrap.fill("Answers, in order: " + " · ".join(q["parts"]), 110))
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def answers(k):
    """The answers practice.py gives for exercise k (for the Python cells below to check against)."""
    return Q[k]["answer"]

def agrees(k, got):
    """True when every value in got is within exercise k's tolerance of its answer."""
    return len(got) == len(answers(k)) and all(right(g, w, t) for g, w, t in zip(got, answers(k), Q[k]["tol"]))

# ---- the course's physics, written out (the same as the practice set's); g = 9.8 m/s² throughout ----
G = 9.8
rad, deg = math.radians, math.degrees
def kmh(v): return v * 3.6                                          # m/s → km/h
def ms(v): return v / 3.6                                           # km/h → m/s
def weight(m): return m * G                                         # W = m g
def momentum(m, v): return m * v                                    # p = m v
def recoil(m_thrown, v_thrown, m_left): return -m_thrown * v_thrown / m_left    # 0 = m₁v₁ + m₂v₂
def coupled(m1, u1, m2, u2): return (m1 * u1 + m2 * u2) / (m1 + m2)
def sym_tension(m, th): return m * G / (2 * math.sin(rad(th)))     # two equal cables at θ above the horizontal
def scale_reading(m, a_up): return m * (G + a_up)                    # N − m g = m a
def slip_angle(mu_s): return deg(math.atan(mu_s))                   # tan θ = μs
def ramp_accel(th, mu_k): return G * (math.sin(rad(th)) - mu_k * math.cos(rad(th)))
def mu_from_ramp(th, a): return (G * math.sin(rad(th)) - a) / (G * math.cos(rad(th)))
def centripetal(m, v, r): return m * v * v / r                     # F = m v²/r
def v_level(mu, r): return math.sqrt(mu * r * G)                    # m v²/r ≤ μs m g
def bank_angle(v, r): return deg(math.atan(v * v / (r * G)))        # tan θ = v²/(r g), no friction
def v_design(th, r): return math.sqrt(r * G * math.tan(rad(th)))
def v_max(r, th, mu):
    t = math.tan(rad(th))
    return math.sqrt(r * G * (t + mu) / (1 - mu * t))
def v_min(r, th, mu):
    t = math.tan(rad(th))
    return 0.0 if t <= mu else math.sqrt(r * G * (t - mu) / (1 + mu * t))

def simulate(deriv, state, t_end, dt=1e-3, stop=None):
    """Fixed-step fourth-order Runge–Kutta, as the course's. deriv(t, s) → ds/dt; returns [(t, state), …] from 0 to
    t_end, or to the first step where stop(t, s) is True, placed on the event (stop.var: the index it watches)."""
    t, s = 0.0, tuple(float(v) for v in state)
    out = [(t, s)]
    for _ in range(int(round(t_end / dt))):
        k1 = deriv(t, s)
        k2 = deriv(t + dt / 2, tuple(a + dt / 2 * b for a, b in zip(s, k1)))
        k3 = deriv(t + dt / 2, tuple(a + dt / 2 * b for a, b in zip(s, k2)))
        k4 = deriv(t + dt, tuple(a + dt * b for a, b in zip(s, k3)))
        s2 = tuple(a + dt / 6 * (b1 + 2 * b2 + 2 * b3 + b4) for a, b1, b2, b3, b4 in zip(s, k1, k2, k3, k4))
        t2 = t + dt
        if stop is not None and stop(t2, s2):
            i = getattr(stop, "var", None)
            if i is not None and s2[i] != s[i]:
                f = (getattr(stop, "level", 0.0) - s[i]) / (s2[i] - s[i])
                t2, s2 = t + f * dt, tuple(a + f * (b - a) for a, b in zip(s, s2))
            out.append((t2, s2))
            return out
        t, s = t2, s2
        out.append((t, s))
    return out

def close(a, b, rel=1e-4):
    return abs(a - b) <= max(rel * max(abs(a), abs(b)), 1e-9)

## Tier 1 · Warm-up

One idea, one step: the net force of two pushes, the first law on a steady van, momentum, weight, F = m a, impulse,
the normal force on a book, the largest static friction, the force a circle needs, and the tilt that needs no
friction.

In [ ]:
practice("P1.1")

In [ ]:
practice("P1.2")

In [ ]:
practice("P1.3")

In [ ]:
practice("P1.4")

In [ ]:
practice("P1.5")

In [ ]:
practice("P1.6")

In [ ]:
practice("P1.7")

In [ ]:
practice("P1.8")

In [ ]:
practice("P1.9")

In [ ]:
practice("P1.10")

### How you'd do this in Python

Each warm-up is one line of arithmetic. Angles need care: Python's `math.sin`, `math.cos` and `math.tan` take
**radians**, not degrees. `math.tan(30)` is the tangent of 30 radians, about −6.41.
Convert first with `math.radians(30)`, and turn an answer from `math.atan` back into degrees with `math.degrees`.

In [ ]:
print(f"math.tan(30)                     = {math.tan(30):.2f}    (30 radians: the trap)")
print(f"math.tan(math.radians(30))       = {math.tan(math.radians(30)):.3f}")
print(f"math.degrees(math.atan(0.5))     = {math.degrees(math.atan(0.5)):.2f}°  (the sliding angle for μs = 0.5)")
assert round(math.tan(30), 2) == -6.41 and round(math.tan(rad(30)), 3) == 0.577 and round(slip_angle(0.5), 2) == 26.57

assert agrees("P1.1", [250 - 180]) and agrees("P1.2", [450])          # steady velocity: the forces balance
assert agrees("P1.3", [momentum(0.06, 30)]) and agrees("P1.4", [weight(75)])
assert agrees("P1.5", [3 / 0.5]) and agrees("P1.6", [50 * 0.2]) and agrees("P1.7", [weight(3)])
assert agrees("P1.8", [0.3 * 200]) and agrees("P1.9", [centripetal(2, 3, 1.5)])
assert agrees("P1.10", [bank_angle(14, 50)])
print(f"P1.10: tan θ = 14² / (50 × 9.8) = {14 ** 2 / (50 * G):.1f} → θ = {bank_angle(14, 50):.2f}°")
print("Every tier 1 answer, worked out in Python, matches the practice set.")

### The puck on ice: slide the friction

A puck leaves a stick at 4 m/s. With no friction at all it never slows: no net force, no change in velocity (the
first law). With friction μk the only horizontal force is μk m g backwards, so it slows at μk g and stops after
v₀ / (μk g) seconds, having slid v₀² / (2 μk g) metres: with μk = 0.05 that's 8.16 s and 16.33 m, with μk = 0.01,
40.82 s and 81.63 m. The curve below is the simulation; the gold dot is where the formula says it stops. Slide μk
down to 0 and watch the stop run away.

In [ ]:
def puck_run(mu, v0=4.0, t_end=60.0, dt=0.01):
    """The puck: x and v, slowed by μk g until it stops (the run ends on the step where v reaches 0)."""
    def deriv(t, s): return (s[1], -mu * G)
    stop = lambda t, s: s[1] < 0; stop.var, stop.level = 1, 0.0
    return simulate(deriv, (0.0, v0), t_end, dt, stop)

for mu_, d_, t_ in ((0.05, 16.33, 8.16), (0.01, 81.63, 40.82)):                       # the video's numbers
    run = puck_run(mu_)
    assert round(run[-1][1][0], 2) == d_ and round(run[-1][0], 2) == t_
    assert close(run[-1][1][0], 4.0 ** 2 / (2 * mu_ * G))

def puck_picture(mu=0.05, v0=4.0):
    run = puck_run(mu, v0)
    t = [p[0] for p in run]; x = [p[1][0] for p in run]
    fig, ax = plt.subplots(figsize=(11.5, 4.2))
    fig.subplots_adjust(left=0.08, right=0.98, top=0.82, bottom=0.16)
    t_max = min(60.0, max(10.0, 1.3 * t[-1])) if mu > 0 else 60.0
    tt = np.linspace(0, t_max, 2)
    ax.plot(tt, v0 * tt, color=DIM, lw=1.4, ls="--", label="no friction: x = v₀ t")
    ax.plot(t, x, color=CHALK, label="the simulated puck")
    if mu > 0:
        T, X = v0 / (mu * G), v0 ** 2 / (2 * mu * G)
        if T <= 60:
            ax.plot([T], [X], "o", color=GOLD, ms=10, label="the formula's stop: v₀²/(2 μk g)")
        title = f"μk = {mu:.2f}: stops after {T:.2f} s, {X:.2f} m from the stick"
    else:
        title = "μk = 0: no net force, so it never slows"
    ax.set_xlim(0, t_max); ax.set_ylim(0, max(x[-1], 10) * 1.25); ax.set_xlabel("time (s)"); ax.set_ylabel("distance (m)")
    ax.legend(frameon=False, loc="lower right")
    fig.suptitle(title, x=0.08, ha="left", fontsize=15, color=GOLD)
    footnote(fig); plt.show()

In [ ]:
interact(puck_picture, mu=FloatSlider(value=0.05, min=0.0, max=0.2, step=0.01, description="μk"),
         v0=FloatSlider(value=4.0, min=1.0, max=10.0, step=0.5, description="v₀ (m/s)"));

## Tier 2 · Core

The level of the "Problems, step by step" chapters: a net force from a speed-up, a ball that bounces back (mind the
sign), recoil, wagons that couple, a lamp on two cables, a lift that accelerates, two blocks pushed together, a crate
already sliding, the angle where a box starts to slide, and the top speed on a flat curve.

In [ ]:
practice("P2.1")

In [ ]:
practice("P2.2")

In [ ]:
practice("P2.3")

In [ ]:
practice("P2.4")

In [ ]:
practice("P2.5")

In [ ]:
practice("P2.6")

In [ ]:
practice("P2.7")

In [ ]:
practice("P2.8")

In [ ]:
practice("P2.9")

In [ ]:
practice("P2.10")

### How you'd do this in Python

**Signs first.** Choose a positive direction and keep it: in P2.2, "away from the wall" is positive, so the ball
arrives at −15 m/s and leaves at +10 m/s, and Δp = m (v − u) = 0.4 × 25. Dropping the minus sign gives 0.4 × 5: the
trap. **Joined bodies**: treat the blocks as one body for the acceleration, then one block alone for the force between
them. Every line below computes one answer with the functions from the first cell and checks it.

In [ ]:
m, u, v, dt = 0.4, -15, 10, 0.02                                      # P2.2: away from the wall is positive
print(f"P2.2: Δp = {m} × ({v} − ({u})) = {m * (v - u):.1f} N s → F = {m * (v - u) / dt:.0f} N"
      f"   (ignoring the sign: {m * (v - abs(u)) / dt:.0f} N, wrong)")
assert agrees("P2.1", [1000 * 25 / 10]) and agrees("P2.2", [m * (v - u) / dt])
assert agrees("P2.3", [abs(recoil(2, 10, 80))]) and agrees("P2.4", [coupled(12000, 3, 8000, 0)])
assert agrees("P2.5", [sym_tension(8, 40)]) and agrees("P2.6", [scale_reading(70, 1.5)])
a_both = 24 / (3 + 5)                                                  # P2.7: both blocks together, then the back one
assert agrees("P2.7", [5 * a_both])
assert agrees("P2.8", [(150 - 0.35 * weight(30)) / 30])
assert agrees("P2.9", [slip_angle(0.35)]) and agrees("P2.10", [v_level(0.6, 45)])
print(f"P2.10: v = √(0.6 × 45 × 9.8) = {v_level(0.6, 45):.2f} m/s = {kmh(v_level(0.6, 45)):.1f} km/h")
print("Every tier 2 answer, worked out in Python, matches the practice set.")

### The catch: slide how soft your hands are

A 0.16 kg ball arrives at 25 m/s. Hands that give a little act like a spring: the ball pushes in, the force grows, and
the ball stops. Stiff hands stop it in 0.01 s; soft hands (a quarter of the stiffness) take 0.02 s. The change in
momentum is the same either way, 0.16 × 25 = 4 N s, and it is the area under the force–time curve. So the longer
stop needs a smaller force: an average of 400 N against 200 N, a peak of 628.3 N against 314.2 N. Slide the stop time
and watch the gold curve get lower and longer, with the same area.

In [ ]:
BALL_M, BALL_V = 0.16, 25.0

def catch_run(t_stop, m=BALL_M, v=BALL_V):
    """Hands as a spring stiff enough to stop the ball in t_stop: k = m (π/(2 t_stop))². Returns k and the run."""
    k = m * (math.pi / (2 * t_stop)) ** 2
    def deriv(t, s): return (s[1], -k * s[0] / m if s[0] > 0 else 0.0)
    stop = lambda t, s: s[1] < 0; stop.var, stop.level = 1, 0.0
    return k, simulate(deriv, (0.0, v), 2 * t_stop, t_stop / 2000, stop)

def catch_numbers(t_stop):
    k, run = catch_run(t_stop)
    t = np.array([p[0] for p in run]); f = k * np.array([p[1][0] for p in run])
    area = float(np.sum((t[1:] - t[:-1]) * (f[1:] + f[:-1]) / 2))      # the area under F–t, by trapezoids
    return t, f, area

for t_stop, peak, avg in ((0.01, 628.3, 400.0), (0.02, 314.2, 200.0)):                 # the video's numbers
    t_, f_, area_ = catch_numbers(t_stop)
    assert round(f_.max(), 1) == peak and close(t_[-1], t_stop, 1e-5) and close(area_, BALL_M * BALL_V, 1e-5)
    assert round(BALL_M * BALL_V / t_[-1], 1) == avg

def catch_picture(stop_ms=20.0):
    fig, ax = plt.subplots(figsize=(11.5, 4.2))
    fig.subplots_adjust(left=0.08, right=0.98, top=0.82, bottom=0.16)
    t1, f1, _ = catch_numbers(0.01)
    ax.plot(1000 * t1, f1, color=DIM, lw=1.6, label="stiff hands: stop in 10 ms")
    t, f, area = catch_numbers(stop_ms / 1000)
    ax.fill_between(1000 * t, f, color=CORAL, alpha=0.25, lw=0)
    ax.plot(1000 * t, f, color=GOLD, label=f"your hands: stop in {stop_ms:.0f} ms")
    ax.set_xlim(0, 52); ax.set_ylim(0, 1500); ax.set_xlabel("time (ms)"); ax.set_ylabel("force on the ball (N)")
    ax.legend(frameon=False, loc="upper right")
    fig.suptitle(f"Peak {f.max():.1f} N, average {BALL_M * BALL_V / t[-1]:.0f} N; the area is still {area:.2f} N s",
                 x=0.08, ha="left", fontsize=15, color=GOLD)
    footnote(fig); plt.show()

In [ ]:
interact(catch_picture, stop_ms=FloatSlider(value=20.0, min=4.0, max=50.0, step=1.0, description="stop (ms)"));

### The slide: tilt the ramp, change the friction

A block slides from rest down a ramp. Along the slope, gravity's part is m g sin θ and kinetic friction pushes back
with μk m g cos θ, so a = g (sin θ − μk cos θ): the mass cancels. At 35° with μk = 0.4 that's 2.41 m/s², and in 2 s
the block slides 4.82 m. The dots are the simulated block every half second. If tan θ is below μk, friction wins
and a block at rest stays put.

In [ ]:
def slide_run(th, mu_k, t_end=2.0):
    a = ramp_accel(th, mu_k)
    return a, simulate(lambda t, s: (s[1], a), (0.0, 0.0), t_end, 0.01)

a_, run_ = slide_run(35, 0.4)
assert round(a_, 2) == 2.41 and round(run_[-1][1][0], 2) == 4.82 and close(run_[-1][1][0], 0.5 * a_ * 2 ** 2)

def slide_picture(theta=35.0, mu_k=0.4):
    a, run = slide_run(theta, mu_k)
    moves = a > 0
    fig, ax = plt.subplots(figsize=(11.5, 4.6))
    fig.subplots_adjust(left=0.05, right=0.98, top=0.84, bottom=0.08)
    L = 12.0                                                           # the ramp: 12 m along the slope
    c, s = math.cos(rad(theta)), math.sin(rad(theta))
    ax.plot([0, L * c, L * c, 0], [L * s, 0, 0, L * s], color=DIM, lw=1.6)
    ax.fill([0, L * c, 0], [L * s, 0, 0], color=GHOST, alpha=0.5, lw=0)
    marks = run[::50]                                                  # every 0.5 s
    for i, (t, st) in enumerate(marks):
        d = st[0] if moves else 0.0
        x, y = d * c, L * s - d * s
        ax.plot([x], [y], "s", color=GOLD if i == len(marks) - 1 else CHALK, ms=12)
        if i % 2 == 0:                                                 # a label every second, under the slope
            ax.annotate(f"{t:.0f} s", (x, y), textcoords="offset points", xytext=(-6, -24), color=DIM, fontsize=11)
    ax.set_aspect("equal"); ax.axis("off")
    ax.set_xlim(-0.6, L * c + 0.6); ax.set_ylim(-0.4, L * s + 0.8)
    if moves:
        title = f"θ = {theta:.0f}°, μk = {mu_k:.2f}: a = {a:.2f} m/s², {run[-1][1][0]:.2f} m in 2 s"
    else:
        title = f"θ = {theta:.0f}°, μk = {mu_k:.2f}: tan θ ≤ μk, so friction holds it (a would be {a:.2f} m/s²)"
    fig.suptitle(title, x=0.05, ha="left", fontsize=15, color=GOLD)
    footnote(fig); plt.show()

In [ ]:
interact(slide_picture, theta=FloatSlider(value=35.0, min=5.0, max=60.0, step=1.0, description="θ (°)"),
         mu_k=FloatSlider(value=0.4, min=0.0, max=1.0, step=0.05, description="μk"));

## Tier 3 · Stretch

Two ideas at once, a formula run backwards, a trap: a lift whose cable just balances its weight, a mass from F and a,
an astronaut's throw worked backwards, a sign on two unequal cables, a lift read from its scale, μk from a measured
acceleration, and a banked curve's design speed and top speed.

In [ ]:
practice("P3.1")

In [ ]:
practice("P3.2")

In [ ]:
practice("P3.3")

In [ ]:
practice("P3.4")

In [ ]:
practice("P3.5")

In [ ]:
practice("P3.6")

In [ ]:
practice("P3.7")

In [ ]:
practice("P3.8")

### How you'd do this in Python

**A free-body diagram is a set of equations.** The sign on two cables at 30° and 45° (P3.4) gives two: sideways,
T₁ cos 30° − T₂ cos 45° = 0; up and down, T₁ sin 30° + T₂ sin 45° = m g. `numpy.linalg.solve` takes the coefficients
as a grid and the right-hand sides as a list, and returns both tensions at once. **Momentum is conserved** in the
skaters' push: the simulation below steps both skaters through the push and checks their total momentum after every
step.

In [ ]:
m, a1, a2 = 20, rad(30), rad(45)
A = np.array([[math.cos(a1), -math.cos(a2)],
              [math.sin(a1), math.sin(a2)]])
T = np.linalg.solve(A, [0, m * G])
print(f"P3.4: T₁ = {T[0]:.2f} N, T₂ = {T[1]:.2f} N (the steeper cable carries more)")
assert agrees("P3.4", list(T))

assert agrees("P3.1", [3])                                             # net force zero: steady speed, any direction
assert agrees("P3.2", [2000 / 0.8]) and agrees("P3.3", [90 * 0.15 / 3])
assert agrees("P3.5", [450 / 50 - G]) and agrees("P3.6", [mu_from_ramp(25, 2.0)])
assert agrees("P3.7", [v_design(8, 150)]) and agrees("P3.8", [v_max(100, 6, 0.3)])
print(f"P3.7: v = √(150 × 9.8 × tan 8°) = {v_design(8, 150):.2f} m/s; P3.8: v_max = {v_max(100, 6, 0.3):.2f} m/s")

# the skaters: 60 kg and 40 kg, pushing apart through a spring-like push (k = 6000 N/m)
m1, m2, k, L = 60.0, 40.0, 6000.0, 1.0
x0 = math.sqrt((m1 * 2.0 ** 2 + m2 * 3.0 ** 2) / k)                    # the squeeze that sends them off at −2 and +3 m/s
def skaters(t, s):
    x1, x2, v1, v2 = s
    f = k * (L - (x2 - x1)) if x2 - x1 < L else 0.0                    # the same push on both, opposite ways
    return (v1, v2, -f / m1, f / m2)
run = simulate(skaters, (0.0, L - x0, 0.0, 0.0), 0.3, 1e-4)
p_max = max(abs(m1 * s[2] + m2 * s[3]) for _, s in run)
print(f"skaters: final velocities {run[-1][1][2]:.3f} and {run[-1][1][3]:.3f} m/s; "
      f"largest total momentum at any step {p_max:.1e} kg m/s")
assert p_max < 1e-9 and close(run[-1][1][2], -2.0, 1e-5) and close(run[-1][1][3], 3.0, 1e-5)
print("Every tier 3 answer, worked out in Python, matches the practice set.")

## The exit ramp: how much should it tilt?

The course's opening question. A highway exit ramp loops round with a radius of **60 m**, and cars take it at
**50 km/h**. The tyres' grip, as a coefficient of static friction (illustrative values): **0.7** on a dry road, **0.4**
on a wet one, **0.1** on ice. The tilt that needs no friction at 50 km/h is steep, and on an icy morning a queue of
stopped cars would slide down it. A gentle tilt, with tan θ = **0.08** (a little under the icy 0.1), holds a stopped
car, and friction does the rest. The next cell works out both tilts, and the safe speeds on a flat ramp and on each
tilt, as a table.

In [ ]:
R, V_KMH, T_GENTLE = 60.0, 50.0, 0.08
GRIP = {"dry": 0.7, "wet": 0.4, "icy": 0.1}
v = ms(V_KMH)
th_free = bank_angle(v, R)                                             # no friction needed at 50 km/h
th_gentle = deg(math.atan(T_GENTLE))                                   # tan θ = 0.08
tilts = {"flat": 0.0, f"the gentle tilt ({th_gentle:.2f}°)": th_gentle, f"the no-friction tilt ({th_free:.2f}°)": th_free}
table = pd.DataFrame({f"{c}: {end}": [kmh(f(R, th, mu)) for th in tilts.values()]
                      for c, mu in GRIP.items() for end, f in (("lowest", v_min), ("top", v_max))},
                     index=list(tilts)).round(1)
print(f"50 km/h = {v:.2f} m/s; v²/(r g) = {v * v / (R * G):.3f}; the no-friction tilt {th_free:.2f}°, "
      f"the gentle one {th_gentle:.2f}°")
display(table)
assert round(th_free, 2) == 18.16 and round(th_gentle, 2) == 4.57 and round(v, 2) == 13.89
assert table.iloc[1].tolist() == [0.0, 79.4, 0.0, 61.5, 0.0, 37.2]                   # the video's numbers
assert table.iloc[0].tolist()[1::2] == [73.0, 55.2, 27.6] and round(table.iloc[2, 4], 1) == 41.0

### The banked road: slide the tilt and the grip

Each bar is the safe speed range on the loop, from the lowest speed (below it, a car slides down the slope towards
the inside) to the top speed (above it, the car slides out). The gold line is the 50 km/h design speed. Start at the
gentle 4.57°: every road holds a stopped car, dry and wet allow 50 km/h, ice tops out at 37.2 km/h. Now push the tilt
to 18.16°: the dry and wet bands stretch, but on ice every car slower than 41 km/h slides in (the red gap). Change the
tilt or the grip, and watch the safe speeds move.

In [ ]:
def band_picture(tilt=th_gentle, dry=0.7, wet=0.4, icy=0.1):
    fig, ax = plt.subplots(figsize=(11.5, 4.2))
    fig.subplots_adjust(left=0.1, right=0.98, top=0.82, bottom=0.16)
    rows = [("dry", dry), ("wet", wet), ("icy", icy)]
    for i, (name, mu) in enumerate(rows):
        lo, hi = kmh(v_min(R, tilt, mu)), kmh(v_max(R, tilt, mu))
        if lo > 0: ax.barh(i, lo, color=CORAL, alpha=0.55, height=0.5)
        ax.barh(i, hi - lo, left=lo, color=SKY, height=0.5)
        ax.text(158, i, f"{lo:.1f} to {hi:.1f} km/h", va="center", ha="right", color=CHALK, fontsize=12)
    ax.axvline(V_KMH, color=GOLD, lw=2.4)
    ax.text(V_KMH + 1, 2.45, "50 km/h", color=GOLD, fontsize=12)
    ax.set_yticks(range(3)); ax.set_yticklabels([f"{n} (μs {mu:.2f})" for n, mu in rows]); ax.invert_yaxis()
    ax.set_xlim(0, 160); ax.set_ylim(2.6, -0.6); ax.set_xlabel("safe speed (km/h)"); ax.set_xticks(range(0, 130, 20))
    fig.suptitle(f"Tilt {tilt:.2f}° on a {R:.0f} m loop: safe speeds on each road", x=0.1, ha="left", fontsize=15,
                 color=GOLD)
    footnote(fig, "computed · our own numbers"); plt.show()

assert [round(kmh(v_max(R, th_gentle, mu)), 1) for mu in GRIP.values()] == [79.4, 61.5, 37.2]   # the slider's start

In [ ]:
interact(band_picture, tilt=FloatSlider(value=th_gentle, min=0.0, max=25.0, step=0.01, description="tilt (°)"),
         dry=FloatSlider(value=0.7, min=0.0, max=1.0, step=0.05, description="μs dry"),
         wet=FloatSlider(value=0.4, min=0.0, max=1.0, step=0.05, description="μs wet"),
         icy=FloatSlider(value=0.1, min=0.0, max=1.0, step=0.05, description="μs icy"));

## Sources
Every exercise, set-up and number here is our own: the exit ramp's grip values and its gentle tilt are illustrative,
not a road standard. The physics is the standard first course in mechanics: Newton's
three laws, momentum and impulse, free-body diagrams and equilibrium, static and kinetic friction, and circular motion
on level and banked roads, with g = 9.8 m/s². The simulations use the classical fourth-order Runge–Kutta method, and
each is checked against the formula the course derives.

In [ ]:
# When this notebook runs beside the course's source files, this cell checks it still matches them; elsewhere it does
# nothing.
import importlib.util, pathlib
_src = pathlib.Path("../../../episodes/ph04_laws_of_motion/practice.py")
if _src.exists():
    _spec = importlib.util.spec_from_file_location("ph04_practice", _src)
    _P = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_P)
    def _flat(a): return [y for x in a for y in _flat(x)] if isinstance(a, (tuple, list)) else [a]
    for _e in _P.EX:
        _q = Q[_e["id"]]
        assert (_q["q"], _q["working"], _q["function"], _q["tier"]) == (_e["q"], _e["working"], _e["function"], _e["tier"]), _e["id"]
        assert len(_q["answer"]) == len(_flat(_e["answer"])) and all(
            abs(a - float(b)) < 1e-9 for a, b in zip(_q["answer"], _flat(_e["answer"]))), _e["id"]
    assert len(_P.EX) == len(EX)
    assert (R, V_KMH, T_GENTLE, GRIP) == (_P.RAMP["r"], _P.RAMP["v_kmh"], _P.RAMP["t_gentle"], _P.RAMP["mu"])
    assert G == _P.G and close(th_free, _P.RP["th_free"], 1e-12) and close(th_gentle, _P.RP["th_gentle"], 1e-12)
    print("matches the course's practice set")